In [1]:
import pandas as pd
import numpy as np

# 1. Read CSV data extracted from database
df = pd.read_csv('01_extracted_loan_data.csv')

# 2. Show first 5 rows to see data structure
print("--- FIRST 5 ROWS ---")
display(df.head())

# 3. Check data structure info and Missing Values
print("\n--- DATA INFO ---")
df.info()

# 4. Check statistical summary to detect Outliers (unreasonable values)
print("\n--- STATISTICAL SUMMARY ---")
display(df.describe())

--- FIRST 5 ROWS ---


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1



--- DATA INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32566 entries, 0 to 32565
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   person_age             32566 non-null  int64  
 1   person_income          32566 non-null  int64  
 2   person_home_ownership  32566 non-null  object 
 3   person_emp_length      31671 non-null  float64
 4   loan_intent            32566 non-null  object 
 5   loan_grade             32566 non-null  object 
 6   loan_amnt              32566 non-null  int64  
 7   loan_int_rate          29451 non-null  float64
 8   loan_status            32566 non-null  int64  
dtypes: float64(2), int64(4), object(3)
memory usage: 2.2+ MB

--- STATISTICAL SUMMARY ---


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status
count,32566.000000,3.256600e+04,31671.000000,32566.000000,29451.000000,32566.000000
mean,27.738163,6.606345e+04,4.790376,9589.328594,11.011396,0.218203
std,6.347369,6.198697e+04,4.143409,6321.769287,3.240613,0.413032
min,21.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000


In [5]:
# 1. Remove Outliers (Unreasonable Values)
# Take customers with age below 100 years and employment length below 60 years
df_clean = df[(df['person_age'] < 100) & (df['person_emp_length'] < 60)].copy()

# 2. Handle Missing Values (Imputation)
# Fill empty data with Median value (middle value) to avoid damaging the distribution
median_emp_length = df_clean['person_emp_length'].median()
median_int_rate = df_clean['loan_int_rate'].median()

df_clean['person_emp_length'] = df_clean['person_emp_length'].fillna(median_emp_length)
df_clean['loan_int_rate'] = df_clean['loan_int_rate'].fillna(median_int_rate)

# 3. Re-check data condition after cleaning
print("Total rows after cleaning:", len(df_clean))
print("\nCheck New Missing Values:")
display(df_clean.isnull().sum())

Total rows after cleaning: 31664

Check New Missing Values:


person_age               0
person_income            0
person_home_ownership    0
person_emp_length        0
loan_intent              0
loan_grade               0
loan_amnt                0
loan_int_rate            0
loan_status              0
dtype: int64

In [3]:
# 1. Calculate Total Default Rate (NPL - Non Performing Loan) for Overall Portfolio
total_default_rate = df_clean['loan_status'].mean() * 100
print(f"Total Default Rate (NPL): {total_default_rate:.2f}%\n")

# 2. Test Assumption 1: Default Rate based on Home Ownership
print("--- DEFAULT RATE BY HOME OWNERSHIP (%) ---")
home_risk = df_clean.groupby('person_home_ownership')['loan_status'].mean() * 100
display(home_risk.sort_values(ascending=False))

# 3. Test Assumption 2: Default Rate based on Loan Purpose
print("\n--- DEFAULT RATE BY LOAN PURPOSE (%) ---")
intent_risk = df_clean.groupby('loan_intent')['loan_status'].mean() * 100
display(intent_risk.sort_values(ascending=False))

# 4. Export Clean Data for Power BI visualization
df_clean.to_csv('03_cleaned_loan_data.csv', index=False)
print("\n[SUCCESS] File '03_cleaned_loan_data.csv' successfully exported!")

Total Default Rate (NPL): 21.55%

--- DEFAULT RATE BY HOME OWNERSHIP (%) ---


person_home_ownership
RENT        31.080408
OTHER       30.841121
MORTGAGE    12.455081
OWN          6.929461
Name: loan_status, dtype: float64


--- DEFAULT RATE BY LOAN PURPOSE (%) ---


loan_intent
DEBTCONSOLIDATION    28.387989
MEDICAL              26.543419
HOMEIMPROVEMENT      25.555556
PERSONAL             19.478099
EDUCATION            16.966417
VENTURE              14.653929
Name: loan_status, dtype: float64


[SUCCESS] File '03_cleaned_loan_data.csv' successfully exported!


In [4]:
# 1. Read ad-hoc extraction results
df_adhoc = pd.read_csv('ad_hoc_young_risk.csv')

# 2. Count total customers in this segment
total_young_customers = len(df_adhoc)

# 3. Calculate default percentage (NPL)
adhoc_npl = df_adhoc['loan_status'].mean() * 100

print(f"Total Customers in This Segment: {total_young_customers} people")
print(f"NPL for Young Customers (<= 25 years) & High Interest (>15%): {adhoc_npl:.2f}%")

Total Customers in This Segment: 1595 people
NPL for Young Customers (<= 25 years) & High Interest (>15%): 60.69%
